# Welcome to the Data Assimilation Cycling tutorial
***
n this tutorial we will explore the Local Ensemble Transform Kalman Filter (LETKF) in a cycling data assimilation framework using the quasi-geostrophic model. It is strongly recommended that you complete the single-cycle LETKF tutorial first, as we will build upon those concepts and focus on what is new about cycling. The LETKF cycling system sequentially estimates the atmospheric state by alternating between analysis and forecast steps over multiple cycles, allowing the ensemble to evolve with the dynamical system and observations over time. In the following we first provide an introduction to cycling concepts for DA, followed by data-assimilation cycling experiments.

## Introduction on Cycling

To Do...

### Set Environment Variables and Number of Cycles

In this tutorial, in addition to setting the typical JEDI_BIN and JEDI_EDU variables, we will automate how many cycles are performed using the global variable `ncycles`. By default this is set to 6. 
> NOTE: **If you would like to perform more than 6 cycles, you will need to first lengthen the truth simulation in 0.qg_tutorial_start**. This can be accomplished by opening `qgstart/yamls/generate_truth.yaml`, editing `forecast length: P21D` to something longer, then saving it and rerunning the truth simulation within the **0.qg_tutorial_start** tutorial

In [ ]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU
export ncycles=6 # If you want more than 6, you need to extend the truth simulation in 0.qg_tutorial_start!!!

### Step 1: Generate the truth

The JEDI system works with **yaml files**, which are essentially control files that contain commands that execute specific tasks coded in the JEDI system. You will get used to them quickly! We first generate a truth run, with would correspond to the evolution of the true atmosphere. This run will be the basis for all experiments we will do in this 3Dvar exercise. We use from the yaml-files directory the file `generate_truth.yaml`. Note the comments after each line, explaining the contents of each line in the yaml file.

First move to your directory `/home/nonroot/shared/EDU/qgletkf/yamls` and open (double click on the file in the tree) the file `generate_truth.yaml`, which looks like this:

```yaml
forecast length: P18D                             # Run truth model for 18 days
geometry:                                         # Declare the geometry of the model grid points
  nx: 40                                          # number of grid points in zonal direction
  ny: 20                                          # number of grid points in the meridional direction
  depths: [4000.0, 6000.0]                        # Height of 1st and 2nd layer
initial condition:                                # Declare details of initial condition
  date: 2009-12-15T00:00:00Z                      # Date of initial condition
  read_from_file: 0                               # Initial condition not read from file but internally generated
model:                                            # Declare model used
  name: QG                                        # Quasi-geostrophic model
  tstep: PT10M                                    # Time step is 10 min
output:                                           # Declare output
  datadir: qgletkf/output/truth/                  # Directory where output file will be stored
  date: 2009-12-15T00:00:00Z                      # Date of output file
  exp: truth                                      # Experiment to generate truth file
  frequency: PT6H                                 # Output frequency is every 6 hours
  type: fc                                        # Truth comes from running qg model in forecast mode
prints:                                           # Declare output specifics
  frequency: PT3H                                 # Generate output every 3 hours
```

Check what this file does: the file runs the 40 by 20 by 2 (layers) quasi-geostrophic model in a standard configuration. Have a good look at the file, see if it makes sense, and note the simple instructions for this simple problem.


After this yaml file is edited as desired we can generate the truth run via:

You should be able to find the output files in `/home/nonroot/shared/EDU/qgletkf/output/truth`. The directory should have the files
```
truth.fc.2009-12-15T00:00:00Z.PT0S.nc
truth.fc.2009-12-15T00:00:00Z.PT6H.nc
truth.fc.2009-12-15T00:00:00Z.PT12H.nc
truth.fc.2009-12-15T00:00:00Z.PT18H.nc
truth.fc.2009-12-15T00:00:00Z.P1D.nc
truth.fc.2009-12-15T00:00:00Z.P1DT6H.nc
…
truth.fc.2009-12-15T00:00:00Z.P18D.nc
```

You can check that this is true by finding these files in the tree on your left, or list the files in `/home/nonroot/shared/EDU/qgletkf/output/truth` with 

In [ ]:
ls $JEDI_EDU/qgletkf/output/truth

Remember that we set the initial time in the yaml file as 2009-12-15 00:00:00. `PTXX` means the forecast length from the initial time. E.g., the actual time for the file 2009-12-15T00:00:00Z.PT12H.nc will be 2009-12-15 12:00:00. Check that the total run time is 18 days (look at the files in the list), consistent with your yaml file.<br><br>
We can make a plot of the truth run using the plotting scripts in `/home/nonroot/shared/EDU/plots_scripts`.
You need to use it like this:
```bash
python ./plot.py <model name> <diagnostic to plot> <path/to/file> --plotwind --output <path/to/output/NAME>
```
For example to plot the different fields of the truth run at 16 days, you will need the following commands:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields $JEDI_EDU/qgletkf/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotwind --output $JEDI_EDU/qgletkf/output/truth/plots/truth

For more details about the plotting tool, please read [the documentation](https://jointcenterforsatellitedataassimilation-jedi-docs.readthedocs-hosted.com/en/latest/inside/jedi-components/oops/toy-models/toy-models.html#unified-plotting-tool).
<br><br>
This calls python to generate plots for the potential vorticity fields, the stream function fields and the wind fields after 16 days of forecast. You can view these plots in `/home/nonroot/shared/EDU/qgletkf/output/truth/plots/`. The last 2 days will be used as forecasts without data assimilation. These plots will show velocity vectors, if you don't want those you can remove the flag *--plotwind* option in the above instructions.
<br><br>
Now have a look at the output using your favorite .jpg viewer (this can be python). The files `truth_x.jpg` and `truth_q.jpg` contain the stream function (x) and potential vorticity (q) in both layers. These should look something like: 
<br>
<center> <img src="images/truth_x_q.png"/> </center>
<br>
Now we are done with the truth, let's generate the background field in the next step.

### Step 2: Generate the background state

We now generate our best initial guess of the state of the system, the *background state*. We use the following yaml `file genenspert_B_default.yaml`:

```yaml
background error:
  covariance model: QgError                         # Use the pre-defined QG error model
  horizontal_length_scale: 2.2e6                    # Horizontal correlation scale in m
  maximum_condition_number: 1.0e6
  standard_deviation: 1.8e7                         # Standard deviation in m^2/s
  vertical_length_scale: 15000.0                    # Vertical correlation lengthscale in m
  randomization_seed: 3                             # Random seed
forecast length: PT24H                              # forecast for one day
initial condition:
  date: 2009-12-31T00:00:00Z
  filename: qgletkf/output/truth/truth.fc.2009-12-15T00:00:00Z.P15D.nc
members: 20                                          # Generate 20 perturbed state from the truth
model:
  name: QG
  tstep: PT1H
output:
  datadir: qgletkf/output/exp_default/bg
  date: 2009-12-30T00:00:00Z
  exp: bkgd
  frequency: PT3H
  type: ens
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
perturbed variables: [x]                            # Perturb the whole state vector x
```

This file perturbs the state variables `[x]` with a random vector drawn from the QG Error covariance model with a correlation length scale of 2200km and standard deviation 1.8e7 $m^2/s$. Note that the control run is generated by adding perturbations to the truth at 2009-12-31-00:00:00. 
You can create the background by running:

In [ ]:
cd $JEDI_EDU
mkdir -p ./qgletkf/output/exp_cycling/bg
$JEDI_BIN/qg_gen_ens_pert_B.x ./qgletkf/yamls/genenspert_B_cycling.yaml

The output file will be in
`$JEDI_EDU/qgletkf/output/exp_default/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc`. You can have a look at the perturbation fields and compare the control run (the background you just created) and the truth by running:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_default/bg/bkgd.ens.1.2009-12-30T00\:00\:00Z.P1D.nc \
        $JEDI_EDU/qg3Dvar/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --output $JEDI_EDU/qgletkf/output/exp_default/plots/bkgd_error_mem1 --title "background error ens 1"
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_default/bg/bkgd.ens.20.2009-12-30T00\:00\:00Z.P1D.nc \
        $JEDI_EDU/qg3Dvar/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --output $JEDI_EDU/qgletkf/output/exp_default/plots/bkgd_error_mem20 --title "background error ens 20"
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_default/bg/bkgd.ens.10.2009-12-30T00\:00\:00Z.P1D.nc \
        $JEDI_EDU/qg3Dvar/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --output $JEDI_EDU/qgletkf/output/exp_default/plots/bkgd_error_mem10 --title "background error ens 10"

The plots will look something like the following. Note that your files might look different because of different random numbers used, but the statistics, such as typical length scales and amplitudes, should be the same. Displayed are the difference between the background stream function and the true stream function for both layers. <br>We can also look at the difference in meridional velocity. Note the size of the differences.
<br>
<center> <img src="images/bkgd_error_x_v_diff.png"/> </center>
<br>

Our next step is to collect observations. In our simulated experiments we actually have to generate the observations from the truth.

Before generating the observations, we should also take a moment to calculate and examine the ensemble variances of the ensemble that was just created. 

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_ens_variance.x ./qgletkf/yamls/ens_variance.yaml

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_default/bg/variance.diag.2009-12-31T00\:00\:00Z.nc \
        --output $JEDI_EDU/qgletkf/output/exp_default/plots/bkgd_ens_variance --title "background ensemble variance"

In this tutorial we will use two types of observation strategies: one observation at one grid point and 100 observations generated at random locations in the model. We start with the one-observation experiments, using an upper-layer stream function observation.

We need to look at the file `makeobs3d_oneobs.yaml` that contains the settings to generate the observation for our first  experiment. Look at each line and check the comments to see if they make sense to you.
```yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: 2009-12-31T00:00:00Z
  filename: qgletkf/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc
window begin: 2009-12-30T18:00:00Z
window length: PT12H
observations:
  observers:
  - obs operator:
      obs type: Stream               # The observation type is the stream function
    obs space:
      obsdataout:
        engine:
          obsfile: qgletkf/output/exp_default/obs/truth.obs3d.nc
      obs type: Stream
      generate:
        begin: PT6H
        nval: 1
        obs locations:
          lon: [150]                 # List of longitudes for generated observations (only one here)
          lat: [67]                  # List of latitudes for generated observations (only one here)
          z: [7000.0]                # List of heights for generated observations (only one here)
        obs error: 4.0e6             # Observation error standard deviation, in m^2/s
        obs period: PT12H
make obs: true                       # Generate the observations
obs perturbations: true              # Add random  measurements to the observations
```

Note the last line, in which we add a random error to the observation. This is to mimic what happens in the real world where measurement errors are always present. The measurement error is drawn from a Gaussian distribution with standard deviation given in the line with `obs error: 4.0e6`.

It reads observations at time 18:00, so 18 hours after the start of the truth run.

Execute the following command to make this observation:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx3d.x ./qgletkf/yamls/makeobs3d_oneobs.yaml

The output file is in `/home/nonroot/shared/EDU/qgletkf/output/exp_default/obs/truth.obs3d.nc`.
To view this file we can convert it to an easily readable txt file with:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qgletkf/output/exp_default/obs/truth.obs3d.nc \
        --output $JEDI_EDU/qgletkf/output/exp_default/plots/qg_oneobs

And display the content of this text file (which has the location, the observation value and the truth denoted as hofx, pronounced "H of x"):

In [ ]:
cat $JEDI_EDU/qgletkf/output/exp_default/plots/qg_oneobs.txt

We are now in the same situation as a numerical weather prediction center: we have the background and observations and are ready to perform the data assimilation in step 4!

We are now in a position to perform the data assimilation, in this case a 3dvar. Have a look at `letkf.yaml` and see if you can understand every line:

```yaml
window begin: &date_bgn 2009-12-30T18:00:00Z
window length: PT12H

geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]

# update (and use for H(x) state at 00Z
background:
  members from template:
    template:
      states:
      - date: &date_mid 2009-12-31T00:00:00Z
        filename: qgletkf/output/exp_default/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc
    pattern: %mem%
    nmembers: 20

observations:
  observers:
  - obs operator:
      obs type: Stream
    obs space:
      obsdatain:
        engine:
          obsfile: qgletkf/output/exp_default/obs/truth.obs3d.nc
      obsdataout:
        engine:
          obsfile: qgletkf/output/exp_default/obs/letkf_obsdataout.obs3d.nc
      obs type: Stream
    obs error:
      covariance model: diagonal
    obs localizations:
    - localization method: Horizontal Gaspari-Cohn 
      lengthscale: 8.0e6

driver:
  update obs config with geometry info: false
  save prior mean: true
  save posterior mean increment: true

local ensemble DA:
  solver: LETKF
  inflation:
    rtpp: 0.5
    mult: 1.1

output:
  states:
  - datadir: qgletkf/output/exp_default/da
    date: *date_mid
    exp: letkf.%{member}%
    type: an

output mean prior:
  datadir: qgletkf/output/exp_default/da
  exp: letkf
  date: *date_mid
  type: meanprior
  
output increment:
  datadir: qgletkf/output/exp_default/da
  exp: letkf
  date: *date_mid
  type: meaninc

```

Run this code via

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_letkf.x ./qgletkf/yamls/letkf.yaml >& ./qgletkf/output/exp_default/letkf.log
[[ $? -eq 0 ]] && echo "SUCCESS RUN OF LETKF" || echo "Error in run, check letkf.log"

**You have done it! Your first ever data-assimilation run with JEDI!**

There will be four files in `/home/nonroot/shared/EDU/qgletkf/output/exp_default/da`. List them with the command below, do you know what each file represents?

In [ ]:
ls $JEDI_EDU/qgletkf/output/exp_default/da

We will now plot the results using the python plotting script. The first plot is a plot of the increments from the data assimilation. 
Run:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_default/da/letkf.000001.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgletkf/output/exp_default/bg/bkgd.ens.1.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsLocations $JEDI_EDU/qgletkf/output/exp_default/obs/letkf_obsdataout.obs3d.nc \
        --plotwind \
        --output $JEDI_EDU/qgletkf/output/exp_default/plots/letkf_mem1_increment --title "analysis increment mem 1"
#python ./plot.py qg fields \
#        $JEDI_EDU/qgletkf/output/exp_default/da/letkf.000020.an.2009-12-31T00\:00\:00Z.nc \
#        $JEDI_EDU/qgletkf/output/exp_default/bg/bkgd.ens.20.2009-12-30T00\:00\:00Z.P1D.nc \
#        --plotObsLocations $JEDI_EDU/qgletkf/output/exp_default/obs/truth.obs3d.nc \
#        --plotwind \
#        --output $JEDI_EDU/qgletkf/output/exp_default/plots/letkf_mem20_increment --title "analysis increment mem 20"  
#python ./plot.py qg fields \
#        $JEDI_EDU/qgletkf/output/exp_default/da/letkf_meaninc.an.2009-12-31T00\:00\:00Z.nc \
#        $JEDI_EDU/qgletkf/output/exp_default/bg/bkgd.ens.0.2009-12-30T00\:00\:00Z.P1D.nc \
#        --plotObsLocations $JEDI_EDU/qgletkf/output/exp_default/obs/truth.obs3d.nc \
#        --plotwind \
#        --output $JEDI_EDU/qgletkf/output/exp_default/plots/letkf_mem20_increment --title "analysis increment mem 20" 

Your plots should look very much like the following.<br>
<center> <img src="images/3dvar_increment_x_diff.png"/> </center>
<br>
Let's see if this figure make sense, and learn a lot about the working of 3DVar in the process. The figure shows the increment in the observed variable, the stream function. But wait, we only observed one grid point in the upper layer, but we see analysis increments *in a much larger area*. To understand this, remember the 3DVar update equation, which for a linear observation operator $H$ is equal to that of the Kalman filter:<br>

<center>$x_a = x_b + BH^T\left(HBH^T+R \right)^{-1}(y-Hx_b)$</center><br>

Since there is only one observation $R$ is a scalar, the observation error variance. This means that $HBH^T$ also has to be a scalar (and it is indeed). Remember hat $H$ maps the state vector in observation space, so the full 1600 (40 by 20 by 2) dimensional state vector is mapped to a 1-dimensional scalar. This means that $H$ is a matrix with 1 row and 1600 columns, and $HBH^T$ is just a scalar.<br><br>
Hence, the spreading of information over the larger domain has to come from $BH^T$, which is a matrix of size 1600 by 1. Indeed, it maps the scaled innovation $\left(HBH^T+R \right)^{-1}(y-Hx_b)$ back to state space. The covariances between grid points in the prior error covariance $B$ spread this information out in state space. Remember that the spatial length scale in the $B$ matrix is 2200 km, and the grid spacing is ???km, so 2.2 grid points. The figure shows that the spread of information is indeed about twice as large as the correlation length scale specified for the B matrix, as expected for a Gaussian correlation structure.<br><br>
This raises the question is we could set this correlation length scale much larger, for instance to get an update over the whole domain just from one observation. Pushing this a bit, one might argue that satellites are not needed, we just need a set of well chosen surface observations and large length scales in the $B$ matrix. However, that thinking is incorrect. The $B$ matrix should contain the length scales of the physical processes. The relevant processes are the high- and low pressure systems, as seen in the true state, one of the first figures in this tutorial above. Indeed, the length scales of the increment have the length scales seen in that figure, which means that we have set the length scales in the $B$ matrix correctly.<br><br>
The physical reason for this length scale is that a pressure perturbation at a certain grid point will be physically related to other pressure perturbations around it, and the typical length scale of these relations is the length scale that should be used in the $B$ matrix.<br><br>

A question: back to the figure, we only observed a grid point in the upper layer, but there is an increment in the lower layer too. Can you figure out why, perhaps from the yaml file used to generate the 3Dvar solution, or in the yaml file used to generate the background state?<br><br>

Indeed, it is the **vertical_length_scale**, which is set to 15000 m. Given the layer thicknesses of 4000 and 6000 m, the two layers are highly correlated, and an increment in the upper layer will be quite similar in the lower layer.<br><br>

As a final thing we will look at the **magnitude** of the increment. At the observation location we can write, because we have only one observation:
<br>
<center> $x_a = x_b + b\left(b+r \right)^{-1}(y-x_b)$ </center> <br>

in which $b$ is the background error variance at the observed location, $r$ is the variance of the observation error, and we removed the $H$ because we are observing the stream function directly at this grid point, so $H=1$. Using this equation and the values for $b$ from the background error standard deviation ($1.8e7 m^2/s$, so $b = 3.24e14$) and for $r$ from the observation error standard deviation ($4.0e6\;m^2/s$, so $r=1.6e13$), and your values for the observation and the background state at that grid point (estimate the latter from the background error and the truth value in the observation file), check that the value of the stream function increment makes sense.<br><br>

For our set of random numbers we obtained $y=-2.18e8$ and $x_b \approx -2.1e8 - 1.9e7 = -2.29e8$, leading to an estimate analysis increment for the stream function of $0.95*(-2.18e8-(-2.29e8)) = 1.0e7$, while the 3DVar analysis in the figure gives about $1.3e7$. This is consistent given our very rough estimate. (Note that this should be an exact match, and it will be if the exact background and analysis values are used.)
<br><br>
We can also look at how derived variables are affected by these increments in stream function. The next two figures show the zonal and meridional velocity increments.<br>

<center> <img src="images/3dvar_increment_u_v_diff.png"/> </center>
<br><br>
Remember that they are related to the streamfunction through<br>

<center>$u = -\frac{\partial \psi}{\partial y} \;\;\;\;\; v = \frac{\partial \psi}{\partial x}$</center> <br>

Check that signs and orders of magnitude are indeed as you would expect.

While these increments are directly derived from the increments in the stream function, this will be more interesting in a numerical weather prediction model. There the pressure (corresponding to the stream function in a QG model) and the velocities are not necessarily in geostrophic balance; we do have ageostrophic velocities in the real atmosphere, and in our more complete numerical models. Still, the velocity increments will be similar to a pressure observation. Why do you think that is?

The answer is that we know that the winds will be in geostrophic balance to a very large extend, so we will build an approximate geostrophic relation into the $B$ matrix. How this is done is beyond this tutorial, but this is standard practice in 3DVar, and has been for many decades.

The final figure is the potential vorticity increment:<br>
<center> <img src="images/3dvar_increment_q_diff.png"/> </center>
<br><br>

Check if this makes sense to you.

After you have done that, note the anti-cyclonic circulation related to the positive mass anomaly from the stream function increment field.

This concludes the first 3DVar data-assimilation experiment. Please check that you:

1. Understand how to generate a truth run, the background field, observations, and a data assimilation run in JEDI,
2. Understand how the background error covariance matrix determines how observation information is spread around over the model state
3. Get an initial idea of how error magnitudes influence the increments. More about that in the next experiments!

In the next experiments we will change background and observation characteristics to get a better feel for the 3DVar and for the JEDI system.

In the experiment **exp_loc**, we increase the localization length scale, so instead of using the same as in **exp_default** `lengthscale 4.0e6` we will use `lengthscale: 8.0e6`. We will follow the same 4-step procedure as before.<br>
### Step 1: Truth
The truth should be exactly the same, we can skip step 1.<br>

### Step 2: Background
We will use the same background generated from exp_default, so we can copy the files<br>


In [ ]:
cp -r $JEDI_EDU/qgletkf/output/exp_default/bg $JEDI_EDU/qgletkf/output/exp_loc/bg
ls $JEDI_EDU/qgletkf/output/exp_loc/bg

### Step 3: Observations

The observations are the same, so we just can copy them from the default experiment (and verify that it is there):

In [ ]:
cp $JEDI_EDU/qgletkf/output/exp_default/obs/truth.obs3d.nc $JEDI_EDU/qgletkf/output/exp_loc/obs
ls $JEDI_EDU/qgletkf/output/exp_loc/obs

### Step 4: Run the data assimilation:
Copy and paste the yaml file `letkf.yaml`:

In [ ]:
cp $JEDI_EDU/qgletkf/yamls/letkf.yaml $JEDI_EDU/qgletkf/yamls/letkf_loc.yaml
ls $JEDI_EDU/qgletkf/yamls/

You need to open `letkf_loc.yaml` and modify several values:
- `lengthscale` under the `obs localizations` section.
- replace all the occurences of `exp_default` by `exp_loc`: this ensures we are using the correct files for background and observations, and aren't overwriting any previous results.
- rename `exp` from `letkf` to `letkf_loc` in `output` sections at the very end of the yaml file and in the parts that save the increments. Your analysis will be saved as `letkf_loc.2009-12-31T00:00:00Z.nc`.

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_letkf.x ./qgletkf/yamls/letkf_loc.yaml

<br>Before you look at the results, take a moment to think and form an idea of what you expect to see. After that, have a look at the results via the increment figures, generated via:

In [ ]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_loc/da/letkf_loc.000001.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgletkf/output/exp_loc/bg/bkgd.ens.1.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsLocations $JEDI_EDU/qgletkf/output/exp_loc/obs/truth.obs3d.nc \
        --plotwind --output $JEDI_EDU/qgletkf/output/exp_loc/plots/letkf_loc_increment --title "analysis increment"

### Step 5: Run the data assimilation again with reduced ensemble members
Copy and paste the yaml file `letkf_loc.yaml`:

In [ ]:
cp $JEDI_EDU/qgletkf/yamls/letkf_loc.yaml $JEDI_EDU/qgletkf/yamls/letkf_loc_mem2.yaml
ls $JEDI_EDU/qgletkf/yamls/

You need to open `letkf_loc_5mem.yaml` and modify `nmembers` to 5. 

Also edit the `exp` to `letkf_loc_mem5` in the `output` and `output increments` section to avoid overwriting the other localization experiment

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_letkf.x ./qgletkf/yamls/letkf_loc_mem2.yaml > ./qgletkf/yamls/mem2.log
[[ $? -eq 0 ]] && echo SUCCESS RUN LETKF || echo Error! 

And now plot the increments. Think carefully what you might see, and how it might compare to the LETKF run using 20 members

In [ ]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_loc/da/letkf_loc_mem5.000001.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgletkf/output/exp_loc/bg/bkgd.ens.1.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsLocations $JEDI_EDU/qgletkf/output/exp_loc/obs/truth.obs3d.nc \
        --plotwind --output $JEDI_EDU/qgletkf/output/exp_loc/plots/letkf_loc_nmem5_increment --title "analysis increment"
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_loc/da/letkf_loc_mem5.000002.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgletkf/output/exp_loc/bg/bkgd.ens.2.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsLocations $JEDI_EDU/qgletkf/output/exp_loc/obs/truth.obs3d.nc \
        --plotwind --output $JEDI_EDU/qgletkf/output/exp_loc/plots/letkf_loc_nmem5_mem2_increment --title "analysis increment"

**To help analyze the difference among these experiments with varying ensemble members, let's compute and plot ensemble variance**

First copy the ens_variance.yaml from experiment one and modify for the loc experiment


In [ ]:
cd $JEDI_EDU
cp ./qgletkf/yamls/ens_variance.yaml ./qgletkf/yamls/ens_variance_loc.yaml
ls ./qgletkf/yamls/

Next open the file `ens_variance_loc.yaml` and modify each instance of `exp_default` to `exp_loc`. Also modify `exp` to `variance_nmem20`

After that, copy this modified `ens_variance_loc.yaml` to `ens_variance_loc_mem5.yaml` and replace `nmembers` to 5 and `exp` to `variance_nmem5`

In [ ]:
cd $JEDI_EDU
cp ./qgletkf/yamls/ens_variance_loc.yaml ./qgletkf/yamls/ens_variance_loc_nmem5.yaml
cp ./qgletkf/yamls/ens_variance_loc.yaml ./qgletkf/yamls/ens_variance_loc_nmem2.yaml
ls ./qgletkf/yamls/

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_ens_variance.x ./qgletkf/yamls/ens_variance_loc.yaml > ./qgletkf/yamls/ensloc.log
[[ $? -eq 0 ]] && echo SUCCESS RUN ENS_VARIANCE || echo Error! 
$JEDI_BIN/qg_ens_variance.x ./qgletkf/yamls/ens_variance_loc_nmem5.yaml > ./qgletkf/yamls/enslocnmem5.log
[[ $? -eq 0 ]] && echo SUCCESS RUN ENS_VARIANCE || echo Error! 
$JEDI_BIN/qg_ens_variance.x ./qgletkf/yamls/ens_variance_loc_nmem2.yaml > ./qgletkf/yamls/enslocnmem2.log
[[ $? -eq 0 ]] && echo SUCCESS RUN ENS_VARIANCE || echo Error! 

Now plot each variance result and compare for using 5 and 20 members. Try to relate the variance plots to the increment plot differences. Can you see any issues with using a reduced ensemble size?

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_loc/bg/variance_nmem20.diag.2009-12-31T00\:00\:00Z.nc \
        --output $JEDI_EDU/qgletkf/output/exp_loc/plots/bkgd_ens_variance_nmem20 --title "bkg variance 20 members"
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_loc/bg/variance_nmem5.diag.2009-12-31T00\:00\:00Z.nc \
        --output $JEDI_EDU/qgletkf/output/exp_loc/plots/bkgd_ens_variance_nmem5 --title "bkg variance 5 members"
python ./plot.py qg fields \
        $JEDI_EDU/qgletkf/output/exp_loc/bg/variance_nmem2.diag.2009-12-31T00\:00\:00Z.nc \
        --output $JEDI_EDU/qgletkf/output/exp_loc/plots/bkgd_ens_variance_nmem2 --title "bkg variance 2 members"

<br>Study the results and try to answer the following questions:

- Why is the increment area smaller than in the default experiment? Is this what you expected?
- Do the velocities have the same magnitude as in the default experiment? Why, or why not?
- Try to make sure you understand 'everything' before you move on!

This experiment should have:
- Increased your skills in using and changing yaml files
- Given you a better understanding of the influence of the horizontal length scales used in the $B$ matrix.

In this experiment, we decrease the vertical length scale of the background error covariance, so instead of using
in `vertical_length_scale: 15000.0` like in **exp_default** we will use `vertical_length_scale: 5000.0` in **exp_ver_len**.

### Step 1: Truth
The truth should be exactly the same. So, we can skip step 1.

### Step 2: Background
Generate a background state and covariance model<br>
In order to be consistent, when we change the vertical length scale of the background error, we also need to re-generate the control run.
Make a copy of the `genenspert_B_default.yaml` file and check that you have a new file named `genenspert_B_ver_len.yaml` in the `yamls` folder:

In [ ]:
cp $JEDI_EDU/qg3Dvar/yamls/genenspert_B_default.yaml $JEDI_EDU/qg3Dvar/yamls/genenspert_B_ver_len.yaml
ls $JEDI_EDU/qg3Dvar/yamls/

You need to modify this new yaml file (right now it is just a copy of the first one). To do so, open `genenspert_B_hor_len.yaml` (double click on it in the tree of files) and change the value for the `vertical_length_scale`to 5000.0.<br>
Don't forget to change which directory (`datadir` in the yaml file) the background will be written to! You need to use `exp_ver_len`.<br><br>
Execute these commands:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_gen_ens_pert_B.x ./qg3Dvar/yamls/genenspert_B_ver_len.yaml

The output file will be in `/home/nonroot/shared/EDU/qg3Dvar/output/exp_ver_len/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc`

### Step 3: Observations

The observations are still the same, we just can copy them from the default experiment:

In [ ]:
cp $JEDI_EDU/qg3Dvar/output/exp_default/obs/truth.obs3d.nc $JEDI_EDU/qg3Dvar/output/exp_ver_len/obs
ls $JEDI_EDU/qg3Dvar/output/exp_ver_len/obs

### Step 4: Run the data assimilation
Copy and paste the yaml file `3dvar_oneobs_default.yaml`:

In [ ]:
cp $JEDI_EDU/qg3Dvar/yamls/3dvar_oneobs_default.yaml $JEDI_EDU/qg3Dvar/yamls/3dvar_oneobs_ver_len.yaml
ls $JEDI_EDU/qg3Dvar/yamls/

You need to open `3dvar_oneobs_hor_len.yaml` and modify several values:
- `vertical_length_scale` under the `background error` section.
- replace all the occurences of `exp_default` by `exp_ver_len`: this ensures we are using the correct files for background and observations, and aren't overwriting any previous results.
- rename `exp` to `3dvar_ver_len` in `output` section at the very end of the yaml file (your analysis will be saved as `3dvar_ver_len.an.2009-12-31T00:00:00Z.nc`)

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qg3Dvar/yamls/3dvar_oneobs_ver_len.yaml

<br><br>
What do you expect to see in the increments? Compare your ideas with the results using the increment figures, generated via:

In [ ]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qg3Dvar/output/exp_ver_len/da/3dvar_ver_len.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg3Dvar/output/exp_ver_len/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsLocations $JEDI_EDU/qg3Dvar/output/exp_ver_len/obs/truth.obs3d.nc \
        --plotwind --output $JEDI_EDU/qg3Dvar/output/exp_ver_len/plots/3dvar_ver_len_increment --title "analysis increment"

Check the differences between these plots and the default experiment.
- Is the increment in the upper layer the same?
- Why is the increment in the lower layer smaller?

This experiment should have
- Increased your skills in using and changing yaml files
- Given you a better understanding of the influence of the horizontal length scales used in the $B$ matrix.

In this experiment, we increase the magnitude of the background error and the observation error covariance. In the yaml used to generate the control run (i.e., add perturbations to the truth) in **exp_default** we were using `standard_deviation: 1.8e7`. In **exp_std_BR** we will use `standard_deviation: 3.6e7`.

In the yaml to generate the observations, we need to use `obs error: 8.0e6` instead of `obs error: 4.0e6`.

### Step 1: Truth
The truth should be exactly the same. So, we can skip step 1.

### Step 2: Background
Generate a background state and covariance model
In order to be consistent, when we change the vertical length scale of the background error, we also need to re-generate the control run.
Make a copy of the `genenspert_B_default.yaml` file and check that you have a new file named `genenspert_B_std_BR.yaml` in the `yamls` folder:

In [ ]:
cp $JEDI_EDU/qg3Dvar/yamls/genenspert_B_default.yaml $JEDI_EDU/qg3Dvar/yamls/genenspert_B_std_BR.yaml
ls $JEDI_EDU/qg3Dvar/yamls/

You need to modify this new yaml file (right now it is just a copy of the first one). To do so, open `genenspert_B_std_BR.yaml` (double click on it in the tree of files) and change the value for the `standard_deviation` to 3.6e7.<br>
Don't forget to change which directory (`datadir` in the yaml file) the background will be written to! <br><br>
Execute these commands:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_gen_ens_pert_B.x ./qg3Dvar/yamls/genenspert_B_std_BR.yaml

The output file will be in `/home/nonroot/shared/EDU/qg3Dvar/output/exp_std_BR/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc`

### Step 3: Observations

Since we increase the observation error, we need to re-generate the observation as well. Make a copy of the `makeobs3d_oneobs.yaml` file and check that you have a new file named `makeobs3d_oneobs_std_BR.yaml` in the `yamls` folder:

In [ ]:
cp $JEDI_EDU/qg3Dvar/yamls/makeobs3d_oneobs.yaml $JEDI_EDU/qg3Dvar/yamls/makeobs3d_oneobs_std_BR.yaml
ls $JEDI_EDU/qg3Dvar/yamls/

You need to modify this new yaml file (right now it is just a copy of the first one). To do so, open `makeobs3d_oneobs_std_BR.yaml` (double click on it in the tree of files) and change the value for the `obs error` to 2.0e6.
Don't forget to change which directory (in the `obsfile` section) the observations will be written to!

Execute these commands:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx3d.x ./qg3Dvar/yamls/makeobs3d_oneobs_std_BR.yaml

Note that we are using the same random seed as is used in the default experiment. Therefore, the same perturbation is applied to generate the perturbation that is added to the truth to obtain the observation, but the magnitude of this perturbation is divided by 2.

### Step 4: Run the data assimilation:
Copy and paste the yaml file `3dvar_oneobs_default.yaml`:

In [ ]:
cp $JEDI_EDU/qg3Dvar/yamls/3dvar_oneobs_default.yaml $JEDI_EDU/qg3Dvar/yamls/3dvar_oneobs_std_BR.yaml
ls $JEDI_EDU/qg3Dvar/yamls/

You need to open `3dvar_oneobs_std_BR.yaml` and modify several values:
- `standard_deviation` under the `background error section` section.
- replace all the occurences of `exp_default` by `exp_std_BR`: this ensures we are using the correct files for background and observations, and aren't overwriting any previous results.
- rename `exp` to `3dvar_std_BR` in `output` section at the very end of the yaml file (your analysis will be saved as `3dvar_std_BR.an.2009-12-31T00:00:00Z.nc`)

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qg3Dvar/yamls/3dvar_oneobs_std_BR.yaml

<br><br>
Before you look - as always - think about what you would expect to see.<br>
Check the results after plotting the increment figures:

In [ ]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qg3Dvar/output/exp_std_BR/da/3dvar_std_BR.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg3Dvar/output/exp_std_BR/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsLocations $JEDI_EDU/qg3Dvar/output/exp_std_BR/obs/truth.obs3d.nc \
        --plotwind --output $JEDI_EDU/qg3Dvar/output/exp_std_BR/plots/3dvar_std_BR_increment --title "analysis increment"

Compare the results with those from the default experiment.
- Are the shapes the same as in the default experiment?
- Are the magnitudes of the increments the same?
- Why are they different by a factor 2?

This experiment should have
- Increased your skills in using and changing yaml files,
- Given you a better understanding of the relative influence of the $B$ and $R$ matrices in 3DVar (and in data assimilation in general).

# Experiment 1: LETKF Cycling with multiple observations
***

For this cycling experiment, we will use 100 randomly generated observations for each cycle time, to mimic the sort of cycling performed with real observations in operational NWP centers.

### Make Observations for All Cycles
Recall the `makeobs3d_mult_obs.yaml` yaml file from the `qgstart` tutorial:

```yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: 2009-12-31T00:00:00Z
  filename: qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc
window begin: 2009-12-30T18:00:00Z
window length: PT12H
observations:
  observers:
  - obs operator:
      obs type: Stream              # The observation type is the stream function
    obs space:
      obsdataout:
        engine:
          obsfile: qgstart/output/obs/truth_100obs.obs3d.nc
      obs type: Stream
      generate:
        begin: PT6H
        nval: 1
        obs density: 100            # Generate 100 observations at random locations
        obs error: 4.0e6            # Observation error standard deviation, in m^2/s
        obs period: PT12H
make obs: true                      # Generate the observations
obs perturbations: true             # Add random  measurements to the observations
```

We will now need to generate observations at several cycles. Given a cycle length of 24 hours, we will need yaml files for every day from this date above (2009-12-31) until the last cycle (2010-01-05 for `ncycles=6`). This can be done dynamically within a loop.

For the above yaml, what parts will need to be modified for each cycle? Try to think on where you would modify the file before reading below.

The answer is the following
- **date:** Must be incremented for each day
- **filename:** The "P16D" will also need to be incremented for each successive day, to generate obs from different days of the truth
- **window begin:** Must be consistent with date and window length (minus 6 hours of date in this case)
- **obsfile:** create unique obsfile names for different cycles

One tricky aspect of this is how to loop dates. Fortunately, the bash command `date` can help in this instance, where you give it the original date and an increment (in days or hours) and it can output the adjusted date in the format specified.

Take a look at how the loop is structured below. Pay attention to how `curdate` is constructed for each cycle and advanced at the end of the loop to the next cycle, how `curdate_minus6h` is created off of curdate, and how the `forecast_hour` is created using simple bash math addition inside the $((...)) syntax. The `echo`command prints out the results of each variable for each cycle. Confirm this is what you expect.

The `cat` command is one option to allow us to dynamically enter this info into unique yaml files for each cycle. Each file is saved at `./qgcycling/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml` for each cycle (icyc = 1,2,3,4,5,6)


In [ ]:
cd $JEDI_EDU

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   forecast_hour="P$((icyc+15))D"
   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, forecast time = $forecast_hour
   
cat << EOF > ./qgcycling/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: ${curdate}
  filename: qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.${forecast_hour}.nc
window begin: ${curdate_minus6h}
window length: PT12H
observations:
  observers:
  - obs operator:
      obs type: Stream                             # The observation type is the stream function
    obs space:
      obsdataout:
        engine:
          obsfile: qgstart/output/obs/truth_cyc${icyc}.obs3d.nc
      obs type: Stream
      generate:
        begin: PT6H
        nval: 1
        obs density: 100                           # Generate 100 observations at random locations
        obs error: 4.0e6                           # Observation error standard deviation, in m^2/s
        obs period: PT12H
make obs: true                                     # Generate the observations
obs perturbations: true                            # Add random  measurements to the observations
EOF

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

After running, you can verify that there were yaml files created. There should be as many files as there are cycles (set by `ncycles`)

In [ ]:
cd $JEDI_EDU
ls ./qgcycling/yamls/makeobs3d_mult_obs_cyc*yaml

We can now run the `qg_hofx` program as in the qgstart tutorial to create observations for each cycle. Note that the outputted cycle observations are saved in the `qgstart/output/obs` directory, as in the qgstart tutorial. For efficiency, the output of the JEDI program has been redirected to log files, with a simple echo statement to confirm each cycle has successfully run or encountered an error.

In [ ]:
cd $JEDI_EDU
for icyc in $(seq 1 $ncycles); do
    $JEDI_BIN/qg_hofx3d.x ./qgcycling/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml >& ./qgcycling/output/makeobs_cyc${icyc}.log
    if [ $? -ne 0 ]; then
        echo "ERROR! Something went wrong running for cycle $icyc. Check ./qgcycling/output/makeobs_cyc${icyc}.log"
    else
        echo "SUCCESS run for cycle $icyc"
    fi
done

If you encounter any issues, check the output log created. Otherwise, let's quickly verify observations were created for all cycles with an `ls` command:

In [ ]:
cd $JEDI_EDU
ls  ./qgstart/output/obs/truth_cyc*.obs3d.nc

### Construct LETKF and forecast yamls

Look at the template file `qgcycling/yamls/letkf_template.yaml`. We will use another method besides `cat` to construct yaml files for each cycle, using a find and replace of certain template keywords in the letkf_template.yaml file (in this case, capitalized words starting with @ are the keywords to be searched and replaced, such as @DATE_BGN). The bash command `sed` can be used for this purpose. Notice we use the same looping with dates and cycles as in the above for observation generation.

> <u>A note on `bgfile`</u>: In cycles after the first, we use output produced from the `qg_forecast.x` program as the background files. Thus, this name **must** be consistent with the output names defined in the  yaml files for the forecast (see `forecast_template.yaml` to confirm this consistency)

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   # for window begin:
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   # for bgfile:
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")

   if [ $icyc -eq 1 ]; then
      bgfile="qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
   else
      bgfile="qgcycling/output/exp_letkf/da/letkf.%mem%.fc.${curdate_minus1day}.P1D.nc"
   fi

   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, bgfile = $bgfile
   
   sed -e "s|@DATE_BGN|${curdate_minus6h}|g;s|@DATE_ANL|${curdate}|g" \
       -e "s|@CYC|$icyc|g;s|@BACKGROUND_FILE|${bgfile}|g" \
       ./qgcycling/yamls/letkf_template.yaml > ./qgcycling/yamls/letkf_cyc${icyc}.yaml

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

Next, we will generate forecast yamls for each cycle and each ensemble member. In this case, a nested loop is required. See `forecast_template.yaml` for the template of each cycle and member. Since all forecasts come after an analysis is produced, the names of the initial condition (IC) files are all the same, only different by what date the ICs are at. Thuse we have only to replace the ensemble member and IC date template keywords with the looped variables for each.

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   for imem in $(seq 1 20); do
      sed -e "s|%MEMBER%|$(printf "%06d\n" "$imem")|g;s|%MEM%|$imem|g;s|@DATE_IC|${curdate}|g" \
             ./qgcycling/yamls/forecast_template.yaml > ./qgcycling/yamls/forecast_cyc${icyc}_mem${imem}.yaml
   done 

    echo "Cycle $icyc: curdate = $curdate, forecast_cyc${icyc}_mem*.yaml files created!"

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done
  

### Run the cycling!

Everything has been produced now to run the cycling experiments fully. Now we loop one more time through the cycles, running first the LETKF step, followed by the ensemble forecast step using the LETKF analysis. This is all setup from the above yaml files that were created. Try to get an understanding of how all it all connects together before running.

This is very similar in concept to what real operational NWP centers do with their DA cycling setups (though in their case, only one loop is generally done, and namelist/yaml files are created on-the-fly rather than pre-generated as we have done here).

NOTE for 6 cycles this will take at least 2 minutes to complete in total, possibly longer depending on the speed of your local machine.

In [ ]:
cd $JEDI_EDU
for icyc in $(seq 1 $ncycles); do
   start_time=`date +%s` # Needed to calculate time it takes to complete each step
   
   # First run LETKF
   $JEDI_BIN/qg_letkf.x ./qgcycling/yamls/letkf_cyc${icyc}.yaml >& ./qgcycling/output/letkf_cyc${icyc}.log
   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running LETKF for cycle $icyc. Check ./qgcycling/output/letkf_cyc${icyc}.log"
   else
      end_time_letkf=`date +%s`
      echo "SUCCESS LETKF run for cycle $icyc, took $(( end_time_letkf - start_time )) seconds"
   fi
   
   # Then run QG ensemble forecast by looping over each member 
   for imem in $(seq 1 20); do
      $JEDI_BIN/qg_forecast.x ./qgcycling/yamls/forecast_cyc${icyc}_mem${imem}.yaml >& ./qgcycling/output/exp_letkf/forecast_cyc${icyc}_mem${imem}.log
      if [ $? -ne 0 ]; then
         echo "ERROR! Something went wrong running forecast for cycle $icyc, member $imem. Check ./qgcycling/output/exp_letkf/forecast_cyc${icyc}_mem${imem}.log"
      else
         echo "SUCCESS forecast run for cycle $icyc, member $imem"
      fi
   done
   end_time_fcst=`date +%s`
   echo "SUCCESS ensemble forecast run for cycle $icyc, took $(( end_time_fcst - end_time_letkf )) seconds"

done

Let's look at some of the results produced within the `qgcycling/output/exp_letkf/da` directory. We will just list out contents from member 1, but you should confirm that 20 members of output were produced in the aforementioned directory.

In [ ]:
cd $JEDI_EDU
echo "Ensemble Member 1 Analyses (used as ICs for member 1 forecast to the next cycle):"
ls ./qgcycling/output/exp_letkf/da/letkf.000001.an*nc
echo -e "\nEnsemble Member 1 Forecasts (used as backgrounds for each cycle):"
ls ./qgcycling/output/exp_letkf/da/letkf.1.fc*P1D.nc
echo -e "\nEnsemble Mean Analyses:"
ls ./qgcycling/output/exp_letkf/da/letkf.000000.an*nc
echo -e "\nEnsemble Mean Prior and Increments:"
ls ./qgcycling/output/exp_letkf/da/letkf_mean*nc


### Compute RMSE, Plot Sawtooths 
We could, at this point, plot increments, analysis errors, forecast errors, etc. for each cycle. However, it is difficult to get a sense for the overall performance of the LETKF assimilation based off of these alone. In this case, we want to quantify the performance of each cycle and plot them to keep track of the stability of the DA system over time. For this purpose, **we will compute RMSE against the truth for both ensemble mean background and mean analysis files at each cycle**. The python script `compute_rmse_layers.py` has been provided for this purpose. To run this script, you need to provide it the model file to evaluate (either mean prior or mean analysis), the truth file to verify against, which variable to compute RMSE for (x,u,v,q), and the destination text file to output the results into, i.e.,

> python compute_rmse_layers.py <input_model.nc> <input_truth.nc> -v x -o <output_rmse_file.txt> -l \<line_label_in_output_text_file>

In [ ]:
cd $JEDI_EDU
output_filename="./qgcycling/output/exp_letkf/plots/rmse_stream.txt"

# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing rmse for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_rmse_layers.py \
          ./qgcycling/output/exp_letkf/da/letkf_meanprior.an.${curdate}.nc \
          ./qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   # Run for analysis mean file
   python ./plots_scripts/compute_rmse_layers.py \
          ./qgcycling/output/exp_letkf/da/letkf.000000.an.${curdate}.nc \
          ./qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          --v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

Next, the python program `plot_rmse_sawtooth.py` reads in the  rmse output file produced by the loop above, parses the information, and plots the sawtooth of background and analysis RMSE for each cycle

Usage:
> python plot_rmse_sawtooth.py <rmse_file.txt> -o <name_of_sawtooth_file.png>

In [ ]:
cd $JEDI_EDU/plots_scripts

python plot_rmse_sawtooth.py $JEDI_EDU/qgcycling/output/exp_letkf/plots/rmse_stream.txt \
                             -o $JEDI_EDU/qgcycling/output/exp_letkf/plots/rmse_sawtooth.png

### Compute Ensemble Variance
To fully assess the performance of LETKF, we need to compute the ensemble variance and compare it to the RMSE values. Since the ensemble variance is a measure of the forecast uncertainty, in a well-tuned system the mean background error should match the estimated uncertainty (ensemble spread, i.e. square root of ensemble variance) from the ensemble. So we will use the 

In [ ]:
cd $JEDI_EDU/
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")
   if [ $icyc -eq 1 ]; then
      bgfile_mem1="qgstart/output/bg/bkgd.ens.1.${curdate_minus1day}.P1D.nc"
      bgfile="qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
   else
      bgfile_mem1="qgcycling/output/exp_letkf/da/letkf.1.fc.${curdate_minus1day}.P1D.nc"
      bgfile="qgcycling/output/exp_letkf/da/letkf.%mem%.fc.${curdate_minus1day}.P1D.nc"
   fi

   # Build namelist for ens variance JEDI program
   cat << EOF > ./qgcycling/yamls/ens_variance_cyc${icyc}.yaml
background:
  date: $curdate
  filename: $bgfile_mem1
  state variables: [x,q,u,v]
ensemble:
  members from template:
    template:
      date: $curdate
      filename: $bgfile
      state variables: [x,q,u,v]
    pattern: %mem%
    nmembers: 20
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
variance output:
  datadir:  qgcycling/output/exp_letkf/da
  exp: variance
  type: diag
  date: $curdate
EOF
   # Run the ens variance program
   $JEDI_BIN/qg_ens_variance.x ./qgcycling/yamls/ens_variance_cyc${icyc}.yaml \
                            >& ./qgcycling/output/exp_letkf/ens_var_cyc${icyc}.log

   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running qg_ens_variance.x for cycle $icyc. Check ./qgcycling/output/exp_letkf/ens_var_cyc${icyc}.log"
   else
      echo "SUCCESS qg_ens_variance.x run for cycle $icyc"
   fi

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

ls ./qgcycling/output/exp_letkf/da/variance*nc

![sawtooth](images/rmse_sawtooth.png)

Another python script, `compute_ensvar_layers.py`, is provided to compute the 2d-average of ensemble variance for each layer, and output to a text file. It is setup similar to the RMSE computation script.

In [ ]:
cd $JEDI_EDU
output_filename="./qgcycling/output/exp_letkf/plots/ensvar_stream.txt"
# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing average ensemble variance for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_ensvar_layers.py \
          ./qgcycling/output/exp_letkf/da/variance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc}"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

The `plot_rmse_sawtooth_withvar.py` script has been provided to plot the sawtooth with ensemble spread overlaid

In [ ]:
cd $JEDI_EDU/plots_scripts

python plot_rmse_sawtooth_withvar.py $JEDI_EDU/qgcycling/output/exp_letkf/plots/rmse_stream.txt \
                             $JEDI_EDU/qgcycling/output/exp_letkf/plots/ensvar_stream.txt \
                             -o $JEDI_EDU/qgcycling/output/exp_letkf/plots/rmse_sawtooth_withvar.png

![sawtooth_withspread](images/rmse_sawtooth_withvar.png)

Now that ensemble spread is plotted along with the RMSE, try to answer these questions:
- Is the system well-tuned? Why or why not?
- What does ensemble spread below the first guess RMSE indicate?

The second question is an important consideration of ensemble-based DA diagnostics. Ideally, spread and RMSE of the first guess should match. In this case, the ensemble spread is below the first guess RMSE, indicative of so-called **underdispersion**. This term means that the ensemble variance itself does not represent the true uncertainty/error of the first guess forecast. 

This indicates more should be done to tune the system for better performance: optimizing localization scale, tuning the analysis inflations, perhaps even changing the number of ensemble members. The capstone project will explore how to perform some of these tunings, and each time we will examine these statistics to evaluate how to best set these values. Perhaps we can get lower RMSE of first guess forecasts with better-tuned localization, or perhaps the analysis ensemble inflation can help to tune the ensemble forecast spread to better match the true error distribution in later cycles.

## Summary

To Do ...